# FedMamba-SALT: Federated Pre-training Experiment (FedAvg / FedProx)

> **Notebook for running federated self-supervised pre-training across 5 clients × 3 data splits.**

This notebook follows the same structure as the centralized experiment notebook but orchestrates
**federated learning** using `train_fedavg.py`. The `--mu` flag toggles between:
- **FedAvg** (`mu=0.0`): Pure weighted averaging of client models
- **FedProx** (`mu>0`): Adds proximal regularisation to prevent client drift

### Experiment Matrix
| Split | FedAvg (`mu=0`) | FedProx (`mu=0.01`) |
|-------|-----------------|---------------------|
| split_1 | ✓ | ✓ |
| split_2 | ✓ | ✓ |
| split_3 | ✓ | ✓ |

### Key Metrics
- **Per-round loss** (weighted average across 5 clients)
- **Per-client loss** (detect stragglers / divergence)
- **enc_std > 0.1** (no representation collapse)
- **Client loss variance** (convergence alignment)

### Architecture (same as centralized)
- **Teacher**: Frozen MAE ViT-B/16 (85M params, never updated)
- **Student**: Inception-Mamba encoder (384-d, depth-6, ~32M trainable)
- **Loss**: SALT (Centred & Standardised cosine distillation)
- **Masking**: Internal Latent Masking (50% token dropout)

## Section 1: Environment Setup

### 1.1 📁 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.2 📍 Set Paths (⚠️ EDIT THESE)

Configure **four** paths: repo location, dataset root, teacher checkpoint, and the **split** to run.

In [ ]:
# ============================================================
# ⚠️  EDIT THESE PATHS TO MATCH YOUR GOOGLE DRIVE LAYOUT
# ============================================================
DRIVE_REPO    = "/content/drive/MyDrive/fedmamba_salt"
DRIVE_DATASET = "/content/drive/MyDrive/Retina"         # must have train/, test/, labels.csv
DRIVE_CKPT    = "/content/drive/MyDrive/original/retina/split1/retina_pretrain_mae_base_split1_checkpoint-1599.pth"

# ============================================================
# ⚠️  FEDERATED EXPERIMENT SETTINGS
# ============================================================
SPLIT_TYPE  = "split_1"     # Options: split_1, split_2, split_3
N_CLIENTS   = 5
MAX_ROUNDS  = 200
E_EPOCH     = 1             # Local epochs per round
MU          = 0.0           # 0.0 = FedAvg, 0.01 = FedProx
BATCH_SIZE  = 128
LR          = 5e-4

NUM_CLASSES = 2  # binary classification
# ============================================================

ALGO_NAME = "FedProx" if MU > 0 else "FedAvg"
print(f"Algorithm:  {ALGO_NAME} (mu={MU})")
print(f"Split:      {SPLIT_TYPE}")
print(f"Clients:    {N_CLIENTS}")
print(f"Rounds:     {MAX_ROUNDS}")
print(f"E_epoch:    {E_EPOCH}")

### 1.2.b 📦 Copy Dataset Locally (Fixes Google Drive I/O Bottleneck)

In [ ]:
import shutil
import os
import time

LOCAL_DATASET = "/content/Retina_local"

print(f"Copying dataset from {DRIVE_DATASET} to {LOCAL_DATASET}...")
print("This might take a few minutes, but will make training 10x faster and prevent hanging!")

start_time = time.time()
if os.path.exists(LOCAL_DATASET):
    print("Local dataset folder already exists. Clearing it...")
    shutil.rmtree(LOCAL_DATASET)

# Copy the entire dataset folder to the local Colab disk
shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)

# Override the DRIVE_DATASET variable to point to the local copy for the rest of the notebook
DRIVE_DATASET = LOCAL_DATASET

elapsed = time.time() - start_time
print(f"\n✓ Dataset copied successfully in {elapsed:.1f} seconds!")
print(f"New dataset path for training: {DRIVE_DATASET}")

In [ ]:
import os

print(f"Verifying checkpoint path: {DRIVE_CKPT}")
if os.path.exists(DRIVE_CKPT):
    print(f"✓ Checkpoint file found at {DRIVE_CKPT}")
else:
    print(f"✗ Checkpoint file NOT found at {DRIVE_CKPT}")
    parent_dir = os.path.dirname(DRIVE_CKPT)
    if os.path.isdir(parent_dir):
        print(f"  Listing contents of parent directory '{parent_dir}':")
        for item in os.listdir(parent_dir):
            print(f"    - {item}")
    else:
        print(f"  Parent directory '{parent_dir}' does not exist.")

print("Please ensure the DRIVE_CKPT path is absolutely correct and matches a file in your Google Drive.")

### 1.3 — Copy Repo to Local Colab Filesystem

Copying to `/content/` avoids slow Google Drive I/O on every Python import. The dataset stays on Drive — DataLoader handles large sequential reads efficiently.

In [ ]:
import shutil, os

LOCAL_REPO = "/content/fedmamba_salt"
if os.path.exists(LOCAL_REPO):
    shutil.rmtree(LOCAL_REPO)
shutil.copytree(DRIVE_REPO, LOCAL_REPO)
os.chdir(LOCAL_REPO)
print(f"Repo copied to {LOCAL_REPO}")

In [ ]:
import shutil, os
# Copy the MAE checkpoint to the expected local path
CKPT_DIR = os.path.join(LOCAL_REPO, "data", "ckpts")
CKPT_PATH = os.path.join(CKPT_DIR, "mae_vit_base.pth")
os.makedirs(CKPT_DIR, exist_ok=True)
if not os.path.exists(CKPT_PATH):
    shutil.copy2(DRIVE_CKPT, CKPT_PATH)
    size_mb = os.path.getsize(CKPT_PATH) / 1e6
    print(f"Checkpoint copied: {size_mb:.1f} MB")
else:
    print(f"Checkpoint already present: {CKPT_PATH}")

# Define output paths for federated experiments
OUTPUT_DIR = os.path.join(LOCAL_REPO, "outputs", f"{ALGO_NAME.lower()}_{SPLIT_TYPE}")
DRIVE_OUTPUT = os.path.join(DRIVE_REPO, "outputs", f"{ALGO_NAME.lower()}_{SPLIT_TYPE}")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

print(f"Local output:  {OUTPUT_DIR}")
print(f"Drive output:  {DRIVE_OUTPUT}")

In [ ]:
import os

print(f"Contents of local output directory ({OUTPUT_DIR}):")
if os.path.exists(OUTPUT_DIR):
    for f in os.listdir(OUTPUT_DIR):
        print(f"  - {f}")
else:
    print(f"  Directory not found: {OUTPUT_DIR}")

print(f"\nContents of Drive output directory ({DRIVE_OUTPUT}):")
if os.path.exists(DRIVE_OUTPUT):
    for f in os.listdir(DRIVE_OUTPUT):
        print(f"  - {f}")
else:
    print(f"  Directory not found: {DRIVE_OUTPUT}")

print("\nPlease ensure 'ckpt_latest.pth' is present in one of these locations for resuming.")

### 1.4 — Install Dependencies

In [ ]:
%%capture install_output
# Build tools required for compiling CUDA kernels


# Install order matters: causal-conv1d before mamba-ssm
# --no-build-isolation lets the build see the existing PyTorch/CUDA
!pip install causal-conv1d>=1.4.0 --no-build-isolation
!pip install mamba-ssm --no-build-isolation

# Teacher backbone and utilities
!pip install timm==0.3.2
!pip install einops PyYAML matplotlib tqdm pandas seaborn

In [ ]:
# Show any errors from install (the output is captured above)
install_text = install_output.stdout + install_output.stderr
errors = [line for line in install_text.split('\n') if 'error' in line.lower() or 'failed' in line.lower()]
if errors:
    print("⚠️  Install errors detected:")
    for e in errors:
        print(f"  {e}")
else:
    print("✓ All dependencies installed successfully.")

### 1.5 — Patch timm for PyTorch 2.x Compatibility

`timm==0.3.2` imports `torch._six` which was removed in PyTorch 2.0+. This shim **must** run before any `import timm`.

> **If you restart the runtime**, re-run from this cell onwards.

In [ ]:
import collections.abc, sys, types

# Create a shim module that satisfies timm's internal import
mock_six = types.ModuleType("torch._six")
mock_six.container_abcs = collections.abc
sys.modules["torch._six"] = mock_six

print("✓ timm compatibility patch applied.")

### 1.6 — Verify Environment

In [ ]:
import torch, timm

print(f"PyTorch:    {torch.__version__}")
print(f"CUDA:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:        {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {mem_gb:.1f} GB")
else:
    print("⚠️  No GPU detected! Select GPU runtime: Runtime > Change runtime type > A100")
print(f"timm:       {timm.__version__}")

try:
    from mamba_ssm import Mamba
    print(f"mamba-ssm:  ✓ (real CUDA kernels)")
except ImportError as e:
    print(f"mamba-ssm:  ✗ FAILED - {e}")

# Add project to Python path
sys.path.insert(0, LOCAL_REPO)
print(f"\nProject root: {LOCAL_REPO}")

---
## Section 2: Checkpoint Verification

In [ ]:
!python -m tests.verify_checkpoint --ckpt_path "{CKPT_PATH}" 

---
## Section 3: Smoke Tests

In [ ]:
print("=" * 55)
print("  Running all smoke tests...")
print("=" * 55)

!python -m tests.test_teacher
!python -m tests.test_student
!python -m tests.test_loss
!python -m tests.test_end_to_end

print("\n" + "=" * 55)
print("  All smoke tests complete.")
print("=" * 55)

> ⛔ **If any test fails, STOP HERE.** Debug the failing component before proceeding to training.

---
## Section 4: Federated Dataset Inspection

### 4.1 — Verify Client Split Structure

In [ ]:
import os, sys
sys.path.insert(0, LOCAL_REPO)

# Check that the federated split CSVs exist
split_dir = os.path.join(DRIVE_DATASET, f"{N_CLIENTS}_clients", SPLIT_TYPE)
print(f"Looking for client splits in: {split_dir}")
print()

if os.path.isdir(split_dir):
    files = sorted(os.listdir(split_dir))
    total_samples = 0
    for f in files:
        fp = os.path.join(split_dir, f)
        if f.endswith('.csv'):
            with open(fp) as fh:
                n = len([l for l in fh if l.strip()])
            total_samples += n
            print(f"  ✓ {f}: {n} samples")
    print(f"\n  Total across all clients: {total_samples} samples")
else:
    print(f"  ✗ Directory not found: {split_dir}")
    print(f"  Run data/data_split.py to generate federated splits.")

### 4.2 — Client Data Distribution

Visualize the class distribution per client to assess non-IID degree.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

split_dir = os.path.join(DRIVE_DATASET, f"{N_CLIENTS}_clients", SPLIT_TYPE)
labels_path = os.path.join(DRIVE_DATASET, "labels.csv")

# Load global labels
labels = {}
with open(labels_path) as f:
    for line in f:
        parts = line.strip().split(",")
        if len(parts) >= 2:
            labels[parts[0]] = int(float(parts[1]))

# Count per-client class distribution
client_dists = {}
for cid in range(1, N_CLIENTS + 1):
    csv_path = os.path.join(split_dir, f"client_{cid}.csv")
    with open(csv_path) as f:
        fnames = [l.strip().split(",")[0] for l in f if l.strip()]
    class_counts = {}
    for fn in fnames:
        lbl = labels.get(fn, -1)
        class_counts[lbl] = class_counts.get(lbl, 0) + 1
    client_dists[cid] = class_counts

# Plot
all_classes = sorted(set(c for d in client_dists.values() for c in d))
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(N_CLIENTS)
width = 0.8 / len(all_classes)

for i, cls in enumerate(all_classes):
    counts = [client_dists[cid].get(cls, 0) for cid in range(1, N_CLIENTS + 1)]
    ax.bar(x + i * width, counts, width, label=f"Class {cls}")

ax.set_xlabel("Client", fontsize=12)
ax.set_ylabel("Number of Samples", fontsize=12)
ax.set_title(f"Client Data Distribution ({SPLIT_TYPE})", fontsize=14, fontweight='bold')
ax.set_xticks(x + width * len(all_classes) / 2)
ax.set_xticklabels([f"Client {i}" for i in range(1, N_CLIENTS + 1)])
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_client_distribution.png"), dpi=150)
plt.show()

### 4.3 — Visualize Augmentation Pairs (Client 1 Sample)

In [ ]:
from augmentations.medical_aug import DualViewDataset, get_teacher_transform, get_student_transform
from augmentations.retina_dataset import RetinaDataset
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF

# Load client 1 data
split_csv = os.path.join(f"{N_CLIENTS}_clients", SPLIT_TYPE, "client_1.csv")
base_ds = RetinaDataset(
    data_path=DRIVE_DATASET, phase="train",
    split_type="federated", split_csv=split_csv,
)
dual_ds = DualViewDataset(
    base_ds,
    teacher_transform=get_teacher_transform(dataset="retina"),
    student_transform=get_student_transform(dataset="retina"),
)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle(f"Augmentation Pairs (Client 1, {SPLIT_TYPE})", fontsize=14, fontweight='bold')

for i in range(4):
    t_view, s_view = dual_ds[i * 10]
    axes[0, i].imshow(TF.to_pil_image(t_view * 0.5 + 0.5))
    axes[0, i].set_title(f"Teacher #{i}", fontsize=10)
    axes[0, i].axis('off')
    axes[1, i].imshow(TF.to_pil_image(s_view * 0.5 + 0.5))
    axes[1, i].set_title(f"Student #{i}", fontsize=10)
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_augmentation_pairs.png"), dpi=150)
plt.show()

---
## Section 5: Federated Pre-training

This section launches `train_fedavg.py` which orchestrates the full federated loop:
1. For each communication round, each of the 5 clients trains locally for `E_epoch` epochs
2. Client models are aggregated via weighted averaging (FedAvg)
3. If `mu > 0`, a FedProx proximal penalty prevents client drift
4. Global model is broadcast back to all clients

**Resume support**: If the notebook disconnects, re-run from Section 1 through here — `train_fedavg.py` will auto-resume from `ckpt_latest.pth`.

### 5.1 — Launch Federated Training

In [ ]:
!python train_fedavg.py \
    --data_path "{DRIVE_DATASET}" \
    --teacher_ckpt "{CKPT_PATH}" \
    --output_dir "{OUTPUT_DIR}" \
    --split_type {SPLIT_TYPE} \
    --n_clients {N_CLIENTS} \
    --max_rounds {MAX_ROUNDS} \
    --E_epoch {E_EPOCH} \
    --mu {MU} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --num_workers 10 \
    --save_every 10 \
    --device cuda

### 5.2 — Backup Checkpoints to Google Drive

In [ ]:
import shutil

os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy checkpoints and metrics to Drive for persistence
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fp) and (f.endswith('.pth') or f.endswith('.csv')):
        shutil.copy2(fp, os.path.join(DRIVE_OUTPUT, f))
        size = os.path.getsize(fp) / 1e6
        print(f"  ✓ {f} ({size:.1f} MB)")

print(f"\nCheckpoints backed up to: {DRIVE_OUTPUT}")
print("These persist even after Colab disconnects.")

---
## Section 6: Training Diagnostics & Visualization

The federated metrics CSV (`federated_metrics.csv`) contains per-round and per-client metrics.
This section visualises convergence, client alignment, and system health.

### 6.0 — Load Metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import os

METRICS_CSV = os.path.join(OUTPUT_DIR, "federated_metrics.csv")
if not os.path.exists(METRICS_CSV):
    METRICS_CSV = os.path.join(DRIVE_OUTPUT, "federated_metrics.csv")

df = pd.read_csv(METRICS_CSV)
print(f"Loaded {len(df)} rounds from {METRICS_CSV}")
print(f"Columns: {list(df.columns)}")
df.head()

### 6.1 — Global SALT Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(df['round'], df['avg_loss'], color='#2196F3', linewidth=2, label='Weighted Avg Loss')
ax.fill_between(df['round'], df['avg_loss'] * 0.9, df['avg_loss'] * 1.1,
                alpha=0.1, color='#2196F3')

ax.axhline(y=0.3, color='#4CAF50', linestyle='--', alpha=0.7, label='Target (< 0.3)')
ax.set_xlabel('Communication Round', fontsize=12)
ax.set_ylabel('SALT Loss', fontsize=12)
ax.set_title(f'{ALGO_NAME} Global Loss ({SPLIT_TYPE})', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_global_loss.png"), dpi=150)
plt.show()

print(f"Final loss: {df['avg_loss'].iloc[-1]:.4f}")
print(f"Best loss:  {df['avg_loss'].min():.4f} (round {df['avg_loss'].idxmin() + 1})")

### 6.2 — Per-Client Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

client_cols = [c for c in df.columns if c.startswith('client_') and c.endswith('_loss')]
colors = ['#E91E63', '#FF9800', '#4CAF50', '#2196F3', '#9C27B0']

for i, col in enumerate(client_cols):
    color = colors[i % len(colors)]
    ax.plot(df['round'], df[col], linewidth=1.5, alpha=0.8,
            color=color, label=f'Client {i+1}')

# Overlay global average
ax.plot(df['round'], df['avg_loss'], color='black', linewidth=2.5,
        linestyle='--', label='Global Avg', zorder=10)

ax.set_xlabel('Communication Round', fontsize=12)
ax.set_ylabel('SALT Loss', fontsize=12)
ax.set_title(f'Per-Client Loss Curves ({ALGO_NAME}, {SPLIT_TYPE})', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_per_client_loss.png"), dpi=150)
plt.show()

### 6.3 — Client Loss Variance (Convergence Alignment)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

client_cols = [c for c in df.columns if c.startswith('client_') and c.endswith('_loss')]
client_df = df[client_cols]
loss_var = client_df.var(axis=1)
loss_std = client_df.std(axis=1)

ax.plot(df['round'], loss_var, color='#FF5722', linewidth=2, label='Variance')
ax.fill_between(df['round'], 0, loss_var, alpha=0.15, color='#FF5722')

ax.set_xlabel('Communication Round', fontsize=12)
ax.set_ylabel('Loss Variance Across Clients', fontsize=12)
ax.set_title(f'Client Convergence Alignment ({ALGO_NAME}, {SPLIT_TYPE})',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_client_variance.png"), dpi=150)
plt.show()

print(f"Initial variance: {loss_var.iloc[0]:.6f}")
print(f"Final variance:   {loss_var.iloc[-1]:.6f}")
print(f"Reduction:        {(1 - loss_var.iloc[-1] / max(loss_var.iloc[0], 1e-10)) * 100:.1f}%")

### 6.4 — Embedding Std (Collapse Detection)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(df['round'], df['avg_enc_std'], color='#E91E63', linewidth=2, label='Avg enc_std')
ax.axhline(y=0.1, color='#4CAF50', linestyle='--', alpha=0.7, label='Healthy (> 0.1)')
ax.axhline(y=0.02, color='#F44336', linestyle='--', alpha=0.7, label='Collapse (< 0.02)')

ax.set_xlabel('Communication Round', fontsize=12)
ax.set_ylabel('Embedding Std', fontsize=12)
ax.set_title(f'Encoder Embedding Std ({ALGO_NAME}, {SPLIT_TYPE})', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_enc_std.png"), dpi=150)
plt.show()

final_std = df['avg_enc_std'].iloc[-1]
print(f"Final enc_std: {final_std:.4f}  {'✓ HEALTHY' if final_std > 0.1 else '✗ WARNING' if final_std > 0.02 else '✗✗ COLLAPSED'}")

### 6.5 — Per-Round Training Time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(df['round'], df['round_time_s'], color='#607D8B', alpha=0.7, width=1.0)
mean_time = df['round_time_s'].mean()
ax.axhline(y=mean_time, color='#FF5722', linestyle='--',
           label=f"Mean: {mean_time:.1f}s")

ax.set_xlabel('Communication Round', fontsize=12)
ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title('Per-Round Training Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_round_timing.png"), dpi=150)
plt.show()

total_hours = df['round_time_s'].sum() / 3600
print(f"Total training time: {total_hours:.2f} hours")
print(f"Average per round:   {mean_time:.1f}s ({N_CLIENTS} clients x {E_EPOCH} local epochs)")

### 6.6 — GPU Memory Usage

In [ ]:
import torch

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df['round'], df['gpu_mb'] / 1024, color='#3F51B5', linewidth=2, label='Allocated')

if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    ax.axhline(y=total_gb, color='gray', linestyle=':', alpha=0.5,
               label=f'Total VRAM ({total_gb:.0f} GB)')

ax.set_xlabel('Communication Round', fontsize=12)
ax.set_ylabel('GPU Memory (GB)', fontsize=12)
ax.set_title('GPU Memory Usage Over Training', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_gpu_memory.png"), dpi=150)
plt.show()

print(f"Peak GPU memory: {df['gpu_mb'].max():.0f} MB ({df['gpu_mb'].max() / 1024:.1f} GB)")

### 6.7 — Combined Federated Training Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'FedMamba-SALT {ALGO_NAME} Dashboard ({SPLIT_TYPE})', fontsize=16, fontweight='bold')

client_cols = [c for c in df.columns if c.startswith('client_') and c.endswith('_loss')]
colors = ['#E91E63', '#FF9800', '#4CAF50', '#2196F3', '#9C27B0']

# --- Global Loss ---
ax = axes[0, 0]
ax.plot(df['round'], df['avg_loss'], color='#2196F3', linewidth=2)
ax.axhline(y=0.3, color='#4CAF50', linestyle='--', alpha=0.7, label='Target')
ax.set_ylabel('Loss'); ax.set_title('Global SALT Loss')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)

# --- Per-Client Loss ---
ax = axes[0, 1]
for i, col in enumerate(client_cols):
    ax.plot(df['round'], df[col], linewidth=1.2, alpha=0.7, color=colors[i % len(colors)],
            label=f'C{i+1}')
ax.set_title('Per-Client Loss')
ax.legend(fontsize=7, ncol=3); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)

# --- Client Variance ---
ax = axes[0, 2]
client_var = df[client_cols].var(axis=1)
ax.plot(df['round'], client_var, color='#FF5722', linewidth=2)
ax.set_title('Client Loss Variance'); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)

# --- Embedding Std ---
ax = axes[1, 0]
ax.plot(df['round'], df['avg_enc_std'], color='#E91E63', linewidth=2)
ax.axhline(y=0.02, color='#F44336', linestyle='--', label='Collapse')
ax.set_ylabel('Std'); ax.set_title('Enc Std'); ax.set_xlabel('Round')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)

# --- Round Time ---
ax = axes[1, 1]
ax.bar(df['round'], df['round_time_s'], color='#607D8B', alpha=0.7, width=1.0)
ax.set_title('Round Time (s)'); ax.set_xlabel('Round')
ax.grid(True, alpha=0.3, axis='y')

# --- GPU Memory ---
ax = axes[1, 2]
ax.plot(df['round'], df['gpu_mb'] / 1024, color='#3F51B5', linewidth=2)
ax.set_title('GPU Memory (GB)'); ax.set_xlabel('Round')
ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_federated_dashboard.png"), dpi=150, bbox_inches='tight')
plt.show()

### 6.8 — Final Health Check

In [ ]:
import torch
from objectives.salt_loss import embedding_std
from models.inception_mamba import InceptionMambaEncoder

ckpt_path = os.path.join(OUTPUT_DIR, "ckpt_latest.pth")
if not os.path.exists(ckpt_path):
    ckpt_path = os.path.join(DRIVE_OUTPUT, "ckpt_latest.pth")

ckpt = torch.load(ckpt_path, map_location="cpu")
final_round = ckpt['comm_round'] + 1
final_loss = ckpt['loss']

# Load student and compute embedding std on random data
student = InceptionMambaEncoder(patch_size=16, embed_dim=384, depth=6, out_dim=768)
student.load_state_dict(ckpt['student_state_dict'])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
student.to(device)
student.eval()

with torch.no_grad():
    dummy = torch.randn(16, 3, 224, 224).to(device)
    emb = student(dummy)
    std_val = embedding_std(emb)

total_time = df['round_time_s'].sum() / 3600

print("=" * 55)
print(f"  {ALGO_NAME} Training Health Check ({SPLIT_TYPE})")
print("=" * 55)
print(f"  Final round:     {final_round}")
print(f"  Final loss:      {final_loss:.4f}  {'✓ PASS' if final_loss < 0.3 else '✗ FAIL'} (target < 0.3)")
print(f"  Embedding std:   {std_val:.4f}  {'✓ PASS' if std_val > 0.05 else '✗ FAIL'} (target > 0.05)")
print(f"  Peak GPU memory: {df['gpu_mb'].max():.0f} MB ({df['gpu_mb'].max() / 1024:.1f} GB)")
print(f"  Total time:      {total_time:.2f} hours")
print("=" * 55)

---
## Section 7: Linear Probe Evaluation

Freeze the federated pre-trained encoder, attach a single linear layer, train only that layer on labeled data.

### 7.1 — Run Linear Probe

In [ ]:
EVAL_LP_DIR = os.path.join(OUTPUT_DIR, "eval_linear_probe")

# Prioritize local checkpoint, but fall back to Drive backup
local_ckpt_path = os.path.join(OUTPUT_DIR, "ckpt_latest.pth")
drive_ckpt_path = os.path.join(DRIVE_OUTPUT, "ckpt_latest.pth")

if os.path.exists(local_ckpt_path):
    FINAL_CKPT = local_ckpt_path
    print(f"Using local checkpoint: {FINAL_CKPT}")
elif os.path.exists(drive_ckpt_path):
    FINAL_CKPT = drive_ckpt_path
    print(f"Using Drive checkpoint: {FINAL_CKPT}")
else:
    raise FileNotFoundError(f"Checkpoint not found locally ({local_ckpt_path}) or on Drive ({drive_ckpt_path})")

In [ ]:
!python -m eval.linear_probe \
    --encoder_ckpt "{FINAL_CKPT}" \
    --data_path "{DRIVE_DATASET}" \
    --num_classes {NUM_CLASSES} \
    --output_dir "{EVAL_LP_DIR}" \
    --epochs 100 \
    --batch_size 512 \
    --lr 1.0e-3 \
    --mode linear_probe

### 7.2 — Display Confusion Matrix

In [ ]:
from IPython.display import Image as IPImage, display
import glob

cm_files = glob.glob(os.path.join(EVAL_LP_DIR, "*confusion*")) + \
           glob.glob(os.path.join(EVAL_LP_DIR, "**/*confusion*"), recursive=True)
if cm_files:
    for f in cm_files:
        print(f"Confusion matrix: {f}")
        display(IPImage(filename=f))
else:
    print("No confusion matrix found. Check eval output directory.")

---
## Section 8: Full Fine-tuning Evaluation

### 8.1 — Run Full Fine-tuning

In [ ]:
EVAL_FT_DIR = os.path.join(OUTPUT_DIR, "eval_full_finetune")

!python -m eval.linear_probe \
    --encoder_ckpt "{FINAL_CKPT}" \
    --data_path "{DRIVE_DATASET}" \
    --num_classes {NUM_CLASSES} \
    --output_dir "{EVAL_FT_DIR}" \
    --epochs 100 \
    --batch_size 256 \
    --lr 1.0e-4 \
    --mode full_finetune

### 8.2 — Display Confusion Matrix

In [ ]:
from IPython.display import Image as IPImage, display
import glob

cm_files = glob.glob(os.path.join(EVAL_FT_DIR, "*confusion*")) + \
           glob.glob(os.path.join(EVAL_FT_DIR, "**/*confusion*"), recursive=True)
if cm_files:
    for f in cm_files:
        print(f"Confusion matrix: {f}")
        display(IPImage(filename=f))
else:
    print("No confusion matrix found. Check eval output directory.")

### 8.3 — Label Scarcity Robustness Experiment

#### 10% Label Scarcity Fine-tuning

In [ ]:
from IPython.display import Image, display
import shutil

EVAL_FT_10_DIR = os.path.join(OUTPUT_DIR, "eval_ft_10pct")

!python -m eval.linear_probe \
    --encoder_ckpt "{FINAL_CKPT}" \
    --data_path "{DRIVE_DATASET}" \
    --num_classes {NUM_CLASSES} \
    --output_dir "{EVAL_FT_10_DIR}" \
    --epochs 100 \
    --batch_size 256 \
    --lr 1.0e-4 \
    --mode full_finetune \
    --label_fraction 0.10

#### 30% Label Scarcity Fine-tuning

In [ ]:
EVAL_FT_30_DIR = os.path.join(OUTPUT_DIR, "eval_ft_30pct")

!python -m eval.linear_probe \
    --encoder_ckpt "{FINAL_CKPT}" \
    --data_path "{DRIVE_DATASET}" \
    --num_classes {NUM_CLASSES} \
    --output_dir "{EVAL_FT_30_DIR}" \
    --epochs 100 \
    --batch_size 256 \
    --lr 1.0e-4 \
    --mode full_finetune \
    --label_fraction 0.30

#### 70% Label Scarcity Fine-tuning

In [ ]:
EVAL_FT_70_DIR = os.path.join(OUTPUT_DIR, "eval_ft_70pct")

!python -m eval.linear_probe \
    --encoder_ckpt "{FINAL_CKPT}" \
    --data_path "{DRIVE_DATASET}" \
    --num_classes {NUM_CLASSES} \
    --output_dir "{EVAL_FT_70_DIR}" \
    --epochs 100 \
    --batch_size 256 \
    --lr 1.0e-4 \
    --mode full_finetune \
    --label_fraction 0.70

#### Label Scarcity Impact Visualization

In [ ]:
import matplotlib.pyplot as plt
import os
import shutil

# Function to parse accuracy from classification report
def get_accuracy(eval_dir):
    report_path = None
    if os.path.exists(eval_dir):
        for root, dirs, files in os.walk(eval_dir):
            for file in files:
                if file.startswith("classification_report") and file.endswith(".txt"):
                    report_path = os.path.join(root, file)
                    break
    if report_path:
        with open(report_path, 'r') as f:
            for line in f:
                if 'weighted avg' in line:
                    return float(line.split()[3]) * 100
    return None

fractions = [10, 30, 70, 100]
eval_dirs = [EVAL_FT_10_DIR, EVAL_FT_30_DIR, EVAL_FT_70_DIR, EVAL_FT_DIR]
accuracies = []

for frac, edir in zip(fractions, eval_dirs):
    acc = get_accuracy(edir)
    if acc is not None:
        accuracies.append(acc)
        print(f"  {frac}%: {acc:.2f}%")
    else:
        accuracies.append(0)
        print(f"  {frac}%: Not found")

baseline_acc = 81.93

if any(a > 0 for a in accuracies):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(fractions, accuracies, 'o-', color='#2196F3', linewidth=2.5,
            markersize=10, markerfacecolor='white', markeredgewidth=2.5,
            label=f'FedMamba-SALT {ALGO_NAME}')
    ax.axhline(y=baseline_acc, color='#F44336', linestyle='--', linewidth=2,
               label='Fed-MAE Centralized Baseline (100% data)')

    ax.set_title('Impact of Label Scarcity on Fine-tuning Accuracy',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Percentage of Labeled Training Data (%)', fontsize=12)
    ax.set_ylabel('Top-1 Accuracy (%)', fontsize=12)
    ax.set_xticks(fractions)

    min_acc = min(min(accuracies) - 5, 65)
    max_acc = max(max(accuracies) + 5, 85)
    ax.set_ylim(min_acc, max_acc)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='lower right')

    for i, acc in enumerate(accuracies):
        if acc > 0:
            ax.annotate(f"{acc:.2f}%", (fractions[i], acc),
                        textcoords="offset points", xytext=(0, 10),
                        ha='center', fontweight='bold')

    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_DIR, 'plot_label_scarcity_impact.png')
    plt.savefig(plot_path, dpi=150)
    if os.path.exists(DRIVE_OUTPUT):
        shutil.copy2(plot_path, os.path.join(DRIVE_OUTPUT, 'plot_label_scarcity_impact.png'))
    plt.show()
else:
    print("No evaluation results found to plot.")

---
## Section 9: Results Summary & Drive Backup

### 9.1 — Final Results

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
import pandas as pd

# Safely load the dataframe if it doesn't exist in the current session
if 'df' not in locals() and 'df' not in globals():
    METRICS_CSV = os.path.join(OUTPUT_DIR, "federated_metrics.csv")
    if not os.path.exists(METRICS_CSV):
        METRICS_CSV = os.path.join(DRIVE_OUTPUT, "federated_metrics.csv")
    if os.path.exists(METRICS_CSV):
        df = pd.read_csv(METRICS_CSV)
    else:
        df = None
        print("Warning: federated_metrics.csv not found.")

# Ensure variables exist
if 'final_loss' not in locals():
    if df is not None:
        final_loss = df['avg_loss'].iloc[-1]
        std_val = df['avg_enc_std'].iloc[-1]
        total_time = df['round_time_s'].sum() / 3600
    else:
        final_loss = 0.0; std_val = 0.0; total_time = 0.0

# Parse accuracies
if 'EVAL_LP_DIR' not in locals():
    EVAL_LP_DIR = os.path.join(OUTPUT_DIR, "eval_linear_probe")
if 'EVAL_FT_DIR' not in locals():
    EVAL_FT_DIR = os.path.join(OUTPUT_DIR, "eval_full_finetune")

def get_accuracy(eval_dir):
    if os.path.exists(eval_dir):
        for root, dirs, files in os.walk(eval_dir):
            for file in files:
                if file.startswith("classification_report") and file.endswith(".txt"):
                    with open(os.path.join(root, file)) as f:
                        for line in f:
                            if 'weighted avg' in line:
                                return float(line.split()[3]) * 100
    return None

lp_acc = get_accuracy(EVAL_LP_DIR)
ft_acc = get_accuracy(EVAL_FT_DIR)
baseline_acc = 81.93
centralized_acc = 82.23

# Print Text Summary
print("=" * 60)
print(f"  FedMamba-SALT: {ALGO_NAME} Experiment Results ({SPLIT_TYPE})")
print("=" * 60)
print(f"  Algorithm:       {ALGO_NAME} (mu={MU})")
print(f"  Split:           {SPLIT_TYPE}")
print(f"  Clients:         {N_CLIENTS}")
print(f"  Rounds:          {MAX_ROUNDS}")
print(f"  Final SALT loss: {final_loss:.4f}  {'✓' if final_loss < 0.3 else '✗'}")
print(f"  Embedding std:   {std_val:.4f}  {'✓' if std_val > 0.05 else '✗'}")
print(f"  Total time:      {total_time:.2f} hours")
print("-" * 60)
if lp_acc: print(f"  Linear Probe:    {lp_acc:.2f}%")
if ft_acc: print(f"  Full Fine-tune:  {ft_acc:.2f}%")
print(f"  Centralized:     {centralized_acc:.2f}%")
print(f"  Fed-MAE Baseline:{baseline_acc:.2f}%")
if ft_acc:
    deg = centralized_acc - ft_acc
    print(f"  Fed Degradation: {deg:+.2f}% (vs centralized {centralized_acc:.2f}%)")
print("=" * 60)

# Create comparison plot
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 6))

models = []
accuracies = []
plot_colors = []

if lp_acc:
    models.append(f'{ALGO_NAME}\nLinear Probe')
    accuracies.append(lp_acc)
    plot_colors.append('#FFC107')
if ft_acc:
    models.append(f'{ALGO_NAME}\nFull Fine-tune')
    accuracies.append(ft_acc)
    plot_colors.append('#00BCD4')

models.append('Centralized\nFedMamba-SALT')
accuracies.append(centralized_acc)
plot_colors.append('#8BC34A')

models.append('Fed-MAE\nBaseline')
accuracies.append(baseline_acc)
plot_colors.append('#E91E63')

bars = ax.bar(models, accuracies, color=plot_colors, edgecolor='black', linewidth=2, width=0.6)
patterns = ['//', '\\\\', 'xx', '..']
for bar, pattern in zip(bars, patterns):
    bar.set_hatch(pattern)

ax.set_ylim(min(accuracies) - 10, max(accuracies) + 5)
ax.set_ylabel('Top-1 Accuracy (%)', fontsize=14, fontweight='bold')
ax.set_title(f'FedMamba-SALT {ALGO_NAME} vs Baselines ({SPLIT_TYPE})',
             fontsize=16, fontweight='bold')

for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
            f'{acc:.2f}%', ha='center', va='bottom', fontsize=14, fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', boxstyle='round,pad=0.2'))

ax.axhline(baseline_acc, color='#E91E63', linestyle='--', linewidth=2, zorder=0, alpha=0.7)

plt.tight_layout()
summary_plot_path = os.path.join(OUTPUT_DIR, 'plot_final_summary.png')
plt.savefig(summary_plot_path, dpi=150)
if os.path.exists(DRIVE_OUTPUT):
    shutil.copy2(summary_plot_path, os.path.join(DRIVE_OUTPUT, 'plot_final_summary.png'))
plt.show()

### 9.2 — Save All Results to Google Drive

In [ ]:
import shutil

os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Save eval directories
for subdir in ["eval_linear_probe", "eval_full_finetune",
               "eval_ft_10pct", "eval_ft_30pct", "eval_ft_70pct"]:
    src = os.path.join(OUTPUT_DIR, subdir)
    dst = os.path.join(DRIVE_OUTPUT, subdir)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  ✓ Saved {subdir}/")

# Save plots, metrics, and checkpoints
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fp) and (f.endswith('.png') or f.endswith('.csv') or f.endswith('.pth')):
        shutil.copy2(fp, os.path.join(DRIVE_OUTPUT, f))
        print(f"  ✓ Saved {f}")

print(f"\nAll results saved to: {DRIVE_OUTPUT}")
print("These persist even after Colab disconnects.")

---

### Expected `federated_metrics.csv` columns

| Column | Description |
|---|---|
| `round` | Communication round (1-indexed) |
| `avg_loss` | Weighted average SALT loss across all clients |
| `avg_enc_std` | Weighted average encoder embedding std |
| `lr` | Current learning rate |
| `round_time_s` | Wall-clock time for the round (seconds) |
| `gpu_mb` | GPU memory allocated (MB) |
| `client_N_loss` | Per-client SALT loss (one column per client) |